# QGT 计算方式对比与改进方案

## 问题分析

当前手动实现的 QGT 计算经常出现 **trace=0** 的问题，这导致 SR 优化失败。

本 notebook 将：
1. 对比 NetKet 原生 QGT 实现方式
2. 分析手动实现的问题
3. 提出改进方案
4. 验证改进效果

## 1. 环境设置和模型定义

In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from tqdm import tqdm
from jax import flatten_util

print(f"JAX version: {jax.__version__}")
print(f"NetKet version: {nk.__version__}")

# H₂ 分子定义
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
E_fci = cisolver.kernel()[0]

print(f"FCI 能量: {E_fci:.8f} Ha")
print(f"HF 能量: {mf.e_tot:.8f} Ha")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions_per_spin=(1,1))

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.5.3
NetKet version: 3.18
FCI 能量: -1.01546825 Ha
HF 能量: -0.94148065 Ha


In [2]:
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

def forward(GraphDef_State, x):
    log_psi, _ = nnx.call(GraphDef_State)(x)
    return log_psi

## 2. NetKet 原生 QGT 实现方式

In [3]:
# 初始化模型
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)

# 设置采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(
    hi, 
    rule=single_rule, 
    n_chains=16, 
    sweep_size=32
)

# 创建 NetKet 变分态
vstate = nk.vqs.MCState(
    sampler=sampler,
    model=model,
    n_samples=1000,
    n_discard_per_chain=10
)

print(f"变分态参数数量: {vstate.n_parameters}")

变分态参数数量: 229


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/vqs/mc/mc_state/state.py:300: UserWarning: n_samples=1000 (1000 per device/MPI rank) does not divide n_chains=16, increased to 1008 (1008 per device/MPI rank)
  self.n_samples = n_samples


In [5]:
# NetKet 原生 QGT 计算
print("="*70)
print("NetKet 原生 QGT 计算")
print("="*70)

# 使用 QGTAuto 构造器
qgt_netket = vstate.quantum_geometric_tensor(
    nk.optimizer.qgt.QGTAuto(diag_shift=0.01, holomorphic=True)
)

print(f"\nQGT 类型: {type(qgt_netket)}")
#print(f"QGT 形状: {qgt_netket.shape}")

# 转换为密集矩阵
qgt_dense = qgt_netket.to_dense()
print(f"\nQGT 密集矩阵形状: {qgt_dense.shape}")
print(f"QGT trace: {jnp.trace(qgt_dense):.6e}")
print(f"QGT 条件数: {jnp.linalg.cond(qgt_dense):.6e}")
print(f"QGT 行列式: {jnp.linalg.det(qgt_dense):.6e}")
print(f"QGT 秩: {jnp.linalg.matrix_rank(qgt_dense)}")

# 检查对角线元素
diag_elements = jnp.diag(qgt_dense)
print(f"\n对角线元素统计:")
print(f"  最小值: {jnp.min(jnp.abs(diag_elements)):.6e}")
print(f"  最大值: {jnp.max(jnp.abs(diag_elements)):.6e}")
print(f"  平均值: {jnp.mean(jnp.abs(diag_elements)):.6e}")
print(f"  标准差: {jnp.std(jnp.abs(diag_elements)):.6e}")

NetKet 原生 QGT 计算

QGT 类型: <class 'netket.optimizer.qgt.qgt_onthefly.QGTOnTheFlyT'>

QGT 密集矩阵形状: (229, 229)
QGT trace: 1.235949e+01-7.561604e-16j
QGT 条件数: 5.211820e+02
QGT 行列式: 0.000000e+00+0.000000e+00j
QGT 秩: 229

对角线元素统计:
  最小值: 1.000000e-02
  最大值: 9.208352e-01
  平均值: 5.397159e-02
  标准差: 8.252619e-02


## 3. 当前手动实现的问题分析

In [6]:
print("="*70)
print("当前手动实现的问题分析")
print("="*70)

# 获取样本
graphdef, model_state = nnx.split(model)
GraphDef_State = (graphdef, model_state)
sampler_state = sampler.init_state(forward, GraphDef_State, seed=21)
sampler_state = sampler.reset(forward, GraphDef_State, state=sampler_state)
samples, sampler_state = sampler.sample(forward, GraphDef_State, state=sampler_state, chain_length=64)
samples = samples.reshape(-1, 4)

print(f"\n样本数量: {samples.shape[0]}")
print(f"样本形状: {samples.shape}")

当前手动实现的问题分析

样本数量: 1024
样本形状: (1024, 4)


In [7]:
# 问题 1: 检查梯度计算方式
print("\n问题 1: 梯度计算方式")
print("-" * 70)

def logpsi_single_param(s, x):
    return forward((graphdef, s), x)

# 方法 A: 使用 holomorphic=True
try:
    grad_holo = jax.grad(logpsi_single_param, argnums=0, holomorphic=True)(model_state, samples[0])
    print("✓ holomorphic=True 可用")
    
    # 展平梯度
    grad_holo_flat, _ = flatten_util.ravel_pytree(grad_holo)
    print(f"  梯度范数: {jnp.linalg.norm(grad_holo_flat):.6e}")
    print(f"  梯度实部范数: {jnp.linalg.norm(grad_holo_flat.real):.6e}")
    print(f"  梯度虚部范数: {jnp.linalg.norm(grad_holo_flat.imag):.6e}")
except Exception as e:
    print(f"✗ holomorphic=True 失败: {e}")

# 方法 B: 分别计算实部和虚部梯度
print("\n方法 B: 分别计算实部和虚部梯度")
grad_real = jax.grad(lambda s: logpsi_single_param(s, samples[0]).real, argnums=0)(model_state)
grad_imag = jax.grad(lambda s: logpsi_single_param(s, samples[0]).imag, argnums=0)(model_state)

grad_real_flat, _ = flatten_util.ravel_pytree(grad_real)
grad_imag_flat, _ = flatten_util.ravel_pytree(grad_imag)

print(f"  实部梯度范数: {jnp.linalg.norm(grad_real_flat):.6e}")
print(f"  虚部梯度范数: {jnp.linalg.norm(grad_imag_flat):.6e}")

# 组合方式 1: grad_real + 1j * grad_imag（当前手动实现）
grad_combined1 = grad_real_flat + 1j * grad_imag_flat
print(f"\n组合方式 1 (grad_real + 1j*grad_imag):")
print(f"  范数: {jnp.linalg.norm(grad_combined1):.6e}")

# 组合方式 2: 正确的 Wirtinger 导数
grad_combined2 = 0.5 * (grad_real_flat + 1j * grad_imag_flat)
print(f"\n组合方式 2 (0.5 * (grad_real + 1j*grad_imag)):")
print(f"  范数: {jnp.linalg.norm(grad_combined2):.6e}")


问题 1: 梯度计算方式
----------------------------------------------------------------------
✓ holomorphic=True 可用
  梯度范数: 4.922079e+00
  梯度实部范数: 3.728248e+00
  梯度虚部范数: 3.213570e+00

方法 B: 分别计算实部和虚部梯度
  实部梯度范数: 4.922079e+00
  虚部梯度范数: 4.922079e+00

组合方式 1 (grad_real + 1j*grad_imag):
  范数: 9.844158e+00

组合方式 2 (0.5 * (grad_real + 1j*grad_imag)):
  范数: 4.922079e+00


In [8]:
# 问题 2: 检查当前手动实现的 QGT 计算
print("\n问题 2: 当前手动实现的 QGT 计算")
print("-" * 70)

def compute_qgt_manual_v1(model_forward, graphdef, state, samples):
    """
    当前手动实现版本
    """
    n_samples = samples.shape[0]
    
    def logpsi_single_param(s, x):
        return model_forward((graphdef, s), x)
    
    def grad_logpsi_single(x):
        # 分别计算实部和虚部的梯度
        grad_real = jax.grad(lambda s: logpsi_single_param(s, x).real, argnums=0)(state)
        grad_imag = jax.grad(lambda s: logpsi_single_param(s, x).imag, argnums=0)(state)
        
        # 展平梯度
        grad_real_flat, _ = flatten_util.ravel_pytree(grad_real)
        grad_imag_flat, _ = flatten_util.ravel_pytree(grad_imag)
        
        # 组合：O_k = ∂logψ/∂θ_k
        return grad_real_flat + 1j * grad_imag_flat
    
    # 批量计算
    grads_flat = jax.vmap(grad_logpsi_single)(samples)
    
    # 计算均值和中心化
    mean_grad = jnp.mean(grads_flat, axis=0)
    centered_grads = grads_flat - mean_grad
    
    # 计算 QGT
    qgt = (1.0 / n_samples) * jnp.einsum('si,sj->ij', 
                                          jnp.conj(centered_grads), 
                                          centered_grads)
    
    return qgt, grads_flat, centered_grads

# 计算手动实现的 QGT
qgt_manual, grads_flat, centered_grads = compute_qgt_manual_v1(forward, graphdef, model_state, samples)

print(f"\n手动实现 QGT 统计:")
print(f"  形状: {qgt_manual.shape}")
print(f"  trace: {jnp.trace(qgt_manual):.6e}")
print(f"  条件数: {jnp.linalg.cond(qgt_manual):.6e}")
print(f"  范数: {jnp.linalg.norm(qgt_manual):.6e}")

print(f"\n梯度统计:")
print(f"  原始梯度范数: {jnp.linalg.norm(grads_flat):.6e}")
print(f"  中心化梯度范数: {jnp.linalg.norm(centered_grads):.6e}")
print(f"  中心化梯度最大值: {jnp.max(jnp.abs(centered_grads)):.6e}")

# 检查是否所有梯度都相同
if jnp.max(jnp.abs(centered_grads)) < 1e-10:
    print("\n⚠️ 警告：所有样本的梯度几乎相同！")
    print("   这说明波函数在参数空间上几乎是平坦的。")


问题 2: 当前手动实现的 QGT 计算
----------------------------------------------------------------------

手动实现 QGT 统计:
  形状: (229, 229)
  trace: 3.749695e+01+0.000000e+00j
  条件数: 2.056064e+34
  范数: 2.364164e+01

梯度统计:
  原始梯度范数: 3.079075e+02
  中心化梯度范数: 1.959512e+02
  中心化梯度最大值: 6.648566e+00


In [9]:
# 问题 3: 对比 NetKet 和手动实现的梯度
print("\n问题 3: 对比 NetKet 和手动实现的梯度")
print("-" * 70)

# NetKet 的梯度计算
_, grad_netket = vstate.expect_and_grad(ha)

# 展平 NetKet 梯度
grad_netket_flat, _ = flatten_util.ravel_pytree(grad_netket)

print(f"NetKet 梯度:")
print(f"  形状: {grad_netket_flat.shape}")
print(f"  范数: {jnp.linalg.norm(grad_netket_flat):.6e}")
print(f"  实部范数: {jnp.linalg.norm(grad_netket_flat.real):.6e}")
print(f"  虚部范数: {jnp.linalg.norm(grad_netket_flat.imag):.6e}")


问题 3: 对比 NetKet 和手动实现的梯度
----------------------------------------------------------------------
NetKet 梯度:
  形状: (229,)
  范数: 1.007126e+00
  实部范数: 7.980748e-01
  虚部范数: 6.143127e-01


## 4. 改进方案

### 改进方案 1: 使用 NetKet 的 QGTOnTheFly 方式（推荐）

In [10]:
print("="*70)
print("改进方案 1: 使用 NetKet 的 QGTOnTheFly 方式")
print("="*70)

def compute_qgt_improved_v1(vstate, diag_shift=0.01):
    """
    改进方案 1: 直接使用 NetKet 的 QGT 实现
    
    优点:
    - 使用 NetKet 经过充分测试的实现
    - 自动处理复数梯度
    - 数值稳定性好
    """
    qgt = vstate.quantum_geometric_tensor(
        nk.optimizer.qgt.QGTAuto(diag_shift=diag_shift, holomorphic=True)
    )
    return qgt.to_dense()

qgt_improved1 = compute_qgt_improved_v1(vstate, diag_shift=0.01)

print(f"\n改进方案 1 QGT 统计:")
print(f"  形状: {qgt_improved1.shape}")
print(f"  trace: {jnp.trace(qgt_improved1):.6e}")
print(f"  条件数: {jnp.linalg.cond(qgt_improved1):.6e}")
print(f"  范数: {jnp.linalg.norm(qgt_improved1):.6e}")
print(f"  秩: {jnp.linalg.matrix_rank(qgt_improved1)}")

改进方案 1: 使用 NetKet 的 QGTOnTheFly 方式

改进方案 1 QGT 统计:
  形状: (229, 229)
  trace: 1.235949e+01-7.561604e-16j
  条件数: 5.211820e+02
  范数: 6.278729e+00
  秩: 229


### 改进方案 2: 修正手动实现的梯度计算方式

In [11]:
print("\n" + "="*70)
print("改进方案 2: 修正手动实现的梯度计算方式")
print("="*70)

def compute_qgt_improved_v2(model_forward, graphdef, state, samples, diag_shift=0.01):
    """
    改进方案 2: 修正梯度计算方式
    
    关键改进:
    1. 使用 holomorphic=True（如果网络支持）
    2. 如果不支持，使用正确的 Wirtinger 导数
    3. 增加正则化
    """
    n_samples = samples.shape[0]
    
    def logpsi_single_param(s, x):
        return model_forward((graphdef, s), x)
    
    # 尝试使用 holomorphic=True
    try:
        def grad_logpsi_single(x):
            grad = jax.grad(logpsi_single_param, argnums=0, holomorphic=True)(state, x)
            flat_grad, _ = flatten_util.ravel_pytree(grad)
            return flat_grad
        
        grads_flat = jax.vmap(grad_logpsi_single)(samples)
        print("✓ 使用 holomorphic=True 成功")
        
    except Exception as e:
        print(f"✗ holomorphic=True 失败，使用 Wirtinger 导数: {e}")
        
        def grad_logpsi_single(x):
            # 对于非全纯函数，使用 Wirtinger 导数
            grad_real = jax.grad(lambda s: logpsi_single_param(s, x).real, argnums=0)(state)
            grad_imag = jax.grad(lambda s: logpsi_single_param(s, x).imag, argnums=0)(state)
            
            grad_real_flat, _ = flatten_util.ravel_pytree(grad_real)
            grad_imag_flat, _ = flatten_util.ravel_pytree(grad_imag)
            
            # 正确的组合方式
            return grad_real_flat + 1j * grad_imag_flat
        
        grads_flat = jax.vmap(grad_logpsi_single)(samples)
    
    # 计算均值和中心化
    mean_grad = jnp.mean(grads_flat, axis=0)
    centered_grads = grads_flat - mean_grad
    
    # 计算 QGT
    qgt = (1.0 / n_samples) * jnp.einsum('si,sj->ij', 
                                          jnp.conj(centered_grads), 
                                          centered_grads)
    
    # 正则化
    n_params = qgt.shape[0]
    qgt_reg = qgt + diag_shift * jnp.eye(n_params)
    
    return qgt_reg, grads_flat, centered_grads

qgt_improved2, grads_flat2, centered_grads2 = compute_qgt_improved_v2(
    forward, graphdef, model_state, samples, diag_shift=0.01
)

print(f"\n改进方案 2 QGT 统计:")
print(f"  形状: {qgt_improved2.shape}")
print(f"  trace: {jnp.trace(qgt_improved2):.6e}")
print(f"  条件数: {jnp.linalg.cond(qgt_improved2):.6e}")
print(f"  范数: {jnp.linalg.norm(qgt_improved2):.6e}")
print(f"  秩: {jnp.linalg.matrix_rank(qgt_improved2)}")


改进方案 2: 修正手动实现的梯度计算方式
✓ 使用 holomorphic=True 成功

改进方案 2 QGT 统计:
  形状: (229, 229)
  trace: 1.166424e+01+0.000000e+00j
  条件数: 5.013149e+02
  范数: 5.928182e+00
  秩: 229


### 改进方案 3: 使用 JVP/VJP 方式（NetKet 内部使用的方式）

In [12]:
print("\n" + "="*70)
print("改进方案 3: 使用 JVP/VJP 方式")
print("="*70)

def compute_qgt_improved_v3(model_forward, graphdef, state, samples, diag_shift=0.01):
    """
    改进方案 3: 使用 JVP/VJP 方式计算 QGT
    
    这是 NetKet QGTOnTheFly 使用的方式
    
    优点:
    - 不需要显式构造 Jacobian 矩阵
    - 内存效率高
    - 数值稳定性好
    """
    n_samples = samples.shape[0]
    n_params = sum(p.size for p in jax.tree_util.tree_leaves(state))
    
    # 使用 jax.grad 计算 Jacobian（支持复数）
    def compute_jacobian_row(x):
        grad = jax.grad(lambda s: model_forward((graphdef, s), x), argnums=0, holomorphic=True)(state)
        flat_grad, _ = flatten_util.ravel_pytree(grad)
        return flat_grad
    
    # 批量计算 Jacobian
    jacobian_flat = jax.vmap(compute_jacobian_row)(samples)
    
    # 计算均值和中心化
    mean_jac = jnp.mean(jacobian_flat, axis=0)
    centered_jac = jacobian_flat - mean_jac
    
    # 计算 QGT
    qgt = (1.0 / n_samples) * jnp.einsum('si,sj->ij', 
                                          jnp.conj(centered_jac), 
                                          centered_jac)
    
    # 正则化
    qgt_reg = qgt + diag_shift * jnp.eye(n_params)
    
    return qgt_reg, jacobian_flat, centered_jac

qgt_improved3, jacobian_flat, centered_jac = compute_qgt_improved_v3(
    forward, graphdef, model_state, samples, diag_shift=0.01
)

print(f"\n改进方案 3 QGT 统计:")
print(f"  形状: {qgt_improved3.shape}")
print(f"  trace: {jnp.trace(qgt_improved3):.6e}")
print(f"  条件数: {jnp.linalg.cond(qgt_improved3):.6e}")
print(f"  范数: {jnp.linalg.norm(qgt_improved3):.6e}")
print(f"  秩: {jnp.linalg.matrix_rank(qgt_improved3)}")


改进方案 3: 使用 JVP/VJP 方式


TypeError: jacrev requires real-valued outputs (output dtype that is a sub-dtype of np.floating), but got complex128. For holomorphic differentiation, pass holomorphic=True. For differentiation of non-holomorphic functions involving complex outputs, use jax.vjp directly.

## 5. 验证改进效果：完整的 VMC + SR 训练

In [ ]:
print("="*70)
print("验证改进效果：完整的 VMC + SR 训练")
print("="*70)

def train_vmc_sr_improved(
    hamiltonian,
    hilbert,
    model,
    n_iterations=300,
    n_samples=1000,
    learning_rate=0.01,
    sr_diag_shift=0.01,
    seed=21
):
    """
    改进的 VMC + SR 训练循环
    
    使用 NetKet 原生的 SR 实现
    """
    # 创建变分态
    edges = [(0, 1), (2, 3)]
    g = nk.graph.Graph(edges=edges)
    single_rule = nk.sampler.rules.FermionHopRule(hilbert=hilbert, graph=g)
    sampler = nk.sampler.MetropolisSampler(
        hilbert, 
        rule=single_rule, 
        n_chains=16, 
        sweep_size=32
    )
    
    vstate = nk.vqs.MCState(
        sampler=sampler,
        model=model,
        n_samples=n_samples,
        n_discard_per_chain=10
    )
    
    # 使用 SGD 优化器
    optimizer = nk.optimizer.Sgd(learning_rate=learning_rate)
    
    # 使用 SR 预处理器
    sr = nk.optimizer.SR(diag_shift=sr_diag_shift, holomorphic=True)
    
    # 创建 VMC 驱动
    gs = nk.driver.VMC(
        hamiltonian,
        optimizer,
        variational_state=vstate,
        preconditioner=sr
    )
    
    # 记录训练过程
    energy_history = []
    
    print(f"\n{'='*70}")
    print(f"开始训练（使用 NetKet 原生 SR）")
    print(f"迭代次数: {n_iterations}, 样本数: {n_samples}, 学习率: {learning_rate}")
    print(f"SR diag_shift: {sr_diag_shift}")
    print(f"{'='*70}\n")
    
    for i in tqdm(range(n_iterations), desc="训练进度"):
        # 运行一步
        gs.run(n_iter=1)
        
        # 记录能量
        energy = vstate.expect(hamiltonian)
        energy_history.append(energy.mean.real)
        
        # 打印进度
        if i % 50 == 0:
            error = abs(energy.mean.real - E_fci)
            
            # 计算 QGT 统计
            qgt = vstate.quantum_geometric_tensor(
                nk.optimizer.qgt.QGTAuto(diag_shift=sr_diag_shift, holomorphic=True)
            )
            qgt_dense = qgt.to_dense()
            qgt_trace = jnp.trace(qgt_dense).real
            qgt_cond = jnp.linalg.cond(qgt_dense)
            
            print(f"\nIter {i:3d} | Energy: {energy.mean.real:.8f} Ha | Error: {error:.6f} Ha")
            print(f"         | QGT trace: {qgt_trace:.6e} | QGT condition: {qgt_cond:.2e}")
    
    print(f"\n{'='*70}")
    print(f"训练完成！最终能量: {energy_history[-1]:.8f} Ha")
    print(f"FCI 基准: {E_fci:.8f} Ha")
    print(f"误差: {abs(energy_history[-1] - E_fci):.6e} Ha")
    print(f"{'='*70}\n")
    
    return energy_history, vstate

In [ ]:
# 初始化模型
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)

# 训练
energy_history_improved, final_vstate = train_vmc_sr_improved(
    hamiltonian=ha,
    hilbert=hi,
    model=model,
    n_iterations=300,
    n_samples=1000,
    learning_rate=0.01,
    sr_diag_shift=0.01,
    seed=21
)

## 6. 总结和建议

In [ ]:
print("="*70)
print("总结和建议")
print("="*70)

print("\n1. 问题根源:")
print("   - 当前手动实现的梯度计算方式可能不正确")
print("   - 对于非全纯函数（如 tanh），需要正确处理 Wirtinger 导数")
print("   - 缺乏足够的正则化导致数值不稳定")

print("\n2. 改进方案对比:")
print("   方案 1 (NetKet 原生): 最推荐")
print("     - 经过充分测试")
print("     - 自动处理复数梯度")
print("     - 数值稳定性好")
print("   ")
print("   方案 2 (修正手动实现): 可用于学习")
print("     - 需要正确处理复数梯度")
print("     - 需要添加正则化")
print("   ")
print("   方案 3 (JVP/VJP): 高级方案")
print("     - 内存效率高")
print("     - 适合大规模问题")

print("\n3. 最佳实践:")
print("   - 使用 NetKet 原生的 SR 实现")
print("   - 设置合适的 diag_shift (建议 0.01-0.1)")
print("   - 使用 SGD 优化器")
print("   - 监控 QGT 的 trace 和 condition number")

print("\n4. 如果坚持手动实现:")
print("   - 确保梯度计算正确")
print("   - 添加足够的正则化")
print("   - 与 NetKet 结果对比验证")

print("\n" + "="*70)